In [1]:
import cv2
import numpy as np
import os
import sys
import shutil
from PIL import Image
from paddle.vision.transforms import functional as F
import re
from ast import literal_eval
from PIL import ImageDraw, ImageFont
import matplotlib.pyplot as plt

In [2]:
source_imgs_dir = 'C:\\Users\\hp\\Desktop\\imgs_line\\'
_source_name_lists = os.listdir(source_imgs_dir)

d:\Anaconda\envs\paddle_env\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [3]:
imgs_name = []
for _item in _source_name_lists:
    if _item.split('.')[-1] in ['jpg', 'png', 'bmp', 'jpeg']:
        imgs_name.append(_item)

In [4]:
# 读取文本文件
_path = os.path.join(source_imgs_dir, 'Label.txt')
with open(_path, 'r', encoding='utf-8') as f:
    contents_label = f.readlines()
_path = os.path.join(source_imgs_dir, 'fileState.txt')
with open(_path, 'r', encoding='utf-8') as f:
    contents_state = f.readlines()


In [5]:
# 把标签中的图片和其索引对应起来，方便后面使用
img_to_label_key = {}
for _i in range(0, len(contents_label)):
    img_to_label_key[(contents_label[_i].split('\t')[0]).split('/')[-1]] = _i

In [6]:
_resize_M = np.array([[0.3, 0],[0.,0.3]], dtype=np.float32)
_trans_vector = np.array([[-2000, -1500]], dtype=np.float32)
_trans_vector2 = np.array([[600, 450]], dtype=np.float32)

_label_file_name = "Label_resize.txt"
with open(os.path.join(source_imgs_dir,_label_file_name),'wb') as f1:
    
    for i in range(0, len(imgs_name)):
        one_label = contents_label[img_to_label_key[imgs_name[i]]]
        src_array_str = re.findall(r'points": (.+?), "difficult', one_label) # 找出点
        _reslist = [literal_eval(_it) for _it in src_array_str] # 点转为列表
        _reslistNumpy = np.array(_reslist)
        _reslistNumpy = _reslistNumpy.astype(np.float32)
        
        points_result = _reslistNumpy + _trans_vector
        points_result = points_result @ _resize_M
        points_result = points_result + _trans_vector2
        
        points_result_int = points_result.astype(np.int32) # 转为int类型
        # 索引是一一对应的，数字数组转字符数组
        points_result_list = points_result_int.tolist()
        points_result_list_str = []
        for _it in points_result_list:
            points_result_list_str.append(str(_it))
        
        # 把原来的点转为模式 
        src_array_str_patten = []
        for _it in src_array_str:
            _new_patten = ''
            for i in range(0, len(_it)):
                if _it[i] == '[' or _it[i] == ']':
                    _new_patten = _new_patten + '\\'
                _new_patten = _new_patten + _it[i]
            src_array_str_patten.append(_new_patten)
            
        # 取代原来的点
        new_label = one_label
        for _src, _dst in zip (src_array_str_patten, points_result_list_str):
            new_label = re.sub(_src, _dst , new_label)

        f1.write(new_label.encode())

In [ ]:
one_label = contents_label[img_to_label_key[imgs_name[0]]]

In [ ]:
print(one_label)

In [ ]:
_res1 = re.findall(r'{(.+?)}', one_label)
# print(_res1)
for _it in _res1:
    print(_it)

In [ ]:
src_array_str = re.findall(r'points": (.+?), "difficult', one_label)
_reslist = [literal_eval(_it) for _it in src_array_str]
for _it in _reslist:
    print(_it)

In [ ]:
_reslistNumpy = np.array(_reslist)
_reslistNumpy = _reslistNumpy.astype(np.float32)
# print(_reslistNumpy.shape)
# print(_reslistNumpy)

In [ ]:
_resize_M = np.array([[0.3, 0],[0.,0.3]], dtype=np.float32)
_trans_vector = np.array([[-2000, -1500]], dtype=np.float32)
_trans_vector2 = np.array([[600, 450]], dtype=np.float32)
# print(_resize_M)

In [ ]:
points_result = _reslistNumpy + _trans_vector
points_result = points_result @ _resize_M
points_result = points_result + _trans_vector2
# print(points_result)

In [ ]:
points_result_int = points_result.astype(np.int32)
# print(points_result_int)

In [ ]:
# 索引是一一对应的
points_result_list = points_result_int.tolist()
points_result_list_str = []
for _it in points_result_list:
    points_result_list_str.append(str(_it))
# print(points_result_int.tolist())

In [ ]:
_ss = ''
_ss = _ss.__add__('a')
_ss = _ss.__add__('b')
print(_ss)

In [ ]:
# print(src_array_str)
src_array_str_patten = []
for _it in src_array_str:
    _new_patten = ''
    for i in range(0, len(_it)):
        if _it[i] == '[' or _it[i] == ']':
            _new_patten = _new_patten + '\\'
        _new_patten = _new_patten + _it[i]
    src_array_str_patten.append(_new_patten)
    

In [ ]:
print(src_array_str_patten)

In [ ]:
# 取代原来的点
new_label = one_label
for _src, _dst in zip (src_array_str_patten, points_result_list_str):
    
    # _src1 = '\\'+_src[0:-1]+'\\'+']'
    print(_src, _dst)
    new_label = re.sub(_src, _dst , new_label)

In [ ]:
print(new_label)

In [ ]:
img = Image.open(os.path.join(source_imgs_dir, imgs_name[0]))
plt.imshow(img)

In [ ]:
draw = ImageDraw.Draw(img)
for rrect in points_result_int:
    for i in range(0, len(rrect)):
        draw.line((rrect[i][0], rrect[i][1], rrect[(i+1)%4][0], rrect[(i+1)%4][1]))
plt.imshow(img)